# 01 - ArchDaily scraping test

## Aim
This notebook tests whether project links and image URLs related to brutalist architecture can be collected from ArchDaily.

## Outputs
- project links
- image URLs
- metadata CSV

In [3]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import time

In [4]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36",
    "Accept-Language": "en-GB,en;q=0.9"
}

In [5]:
search_url = "https://www.archdaily.com/search/projects/categories/brutalism"

resp = requests.get(search_url, headers=HEADERS, timeout=20)
print(resp.status_code)
print(resp.url)
print(resp.text[:500])

200
https://www.archdaily.com/search/projects/categories/brutalism
<!DOCTYPE html>
<html lang="en-US" >
  <head>

    <!-- Google Tag Manager -->
    <script>
      window.dataLayer = window.dataLayer || [];
      window.dataLayer.push({
        platform: "ad",
        appId: "search"
      });

      // Read userId from cookie and add to dataLayer before GTM loads
      function getCookie(name) {
        var nameEQ = name + "=";
        var ca = document.cookie.split(';');
        for (var i = 0; i < ca.length; i++) {
          var c = ca[i];
          while (


In [6]:
def extract_project_links(html: str):
    soup = BeautifulSoup(html, "lxml")
    links = set()

    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/projects/" in href:
            full = urljoin("https://www.archdaily.com", href)
            links.add(full)

    return sorted(links)

In [7]:
project_links = extract_project_links(resp.text)
len(project_links), project_links[:20]

(8,
 ['https://www.archdaily.com/search/projects/categories/commercial-and-offices?ad_source=jv-header&ad_name=hamburger_menu',
  'https://www.archdaily.com/search/projects/categories/cultural-architecture?ad_source=jv-header&ad_name=hamburger_menu',
  'https://www.archdaily.com/search/projects/categories/educational-architecture?ad_source=jv-header&ad_name=hamburger_menu',
  'https://www.archdaily.com/search/projects/categories/hospitality-architecture?ad_source=jv-header&ad_name=hamburger_menu',
  'https://www.archdaily.com/search/projects/categories/interior-design?ad_source=jv-header&ad_name=hamburger_menu',
  'https://www.archdaily.com/search/projects/categories/landscape-and-urbanism?ad_source=jv-header&ad_name=hamburger_menu',
  'https://www.archdaily.com/search/projects/categories/public-architecture?ad_source=jv-header&ad_name=hamburger_menu',
  'https://www.archdaily.com/search/projects/categories/residential-architecture?ad_source=jv-header&ad_name=hamburger_menu'])

In [8]:
df_links = pd.DataFrame({"project_url": project_links})
df_links.to_csv("../data/raw/archdaily_project_links_test.csv", index=False)
df_links.head()

,project_url
0,https://www.archdaily.com/search/projects/cate...
1,https://www.archdaily.com/search/projects/cate...
2,https://www.archdaily.com/search/projects/cate...
3,https://www.archdaily.com/search/projects/cate...
4,https://www.archdaily.com/search/projects/cate...


## Step 2 - Filter valid project links

The initial extraction returned search-related URLs. This step filters the results to keep only valid project pages.

In [1]:
import re
from urllib.parse import urljoin
from bs4 import BeautifulSoup
import requests
import pandas as pd

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36",
    "Accept-Language": "en-GB,en;q=0.9"
}

search_url = "https://www.archdaily.com/search/projects/categories/brutalism"
resp = requests.get(search_url, headers=HEADERS, timeout=20)
print(resp.status_code)

200


In [6]:
def extract_valid_project_links(html: str):
    soup = BeautifulSoup(html, "lxml")
    links = set()

    for a in soup.find_all("a", href=True):
        href = a["href"]
        full = urljoin("https://www.archdaily.com", href)

        # 只要包含 archdaily 主域名、并且路径里有数字段，就先保留
        if full.startswith("https://www.archdaily.com/") and "/search/" not in full:
            parts = full.replace("https://www.archdaily.com/", "").split("/")
            if len(parts) > 0 and parts[0].isdigit():
                links.add(full)

    return sorted(links)

In [7]:
valid_project_links = extract_valid_project_links(resp.text)
len(valid_project_links)
valid_project_links[:20]

[]

In [8]:
df_valid_links = pd.DataFrame({"project_url": valid_project_links})
df_valid_links.to_csv("../data/raw/valid_project_links_test.csv", index=False)
df_valid_links.head()

,project_url


In [9]:
test_url = valid_project_links[0]
test_url

IndexError: list index out of range

In [10]:
from urllib.parse import urljoin
from bs4 import BeautifulSoup

soup = BeautifulSoup(resp.text, "lxml")

all_links = []
for a in soup.find_all("a", href=True):
    href = a["href"]
    full = urljoin("https://www.archdaily.com", href)
    all_links.append(full)

unique_links = sorted(set(all_links))

print("Total unique links:", len(unique_links))
for link in unique_links[:80]:
    print(link)

Total unique links: 112
https://archdaily.com/search/images?ad_source=jv-header&ad_name=main-menu
https://architecturecompetitions.com/unbuilt2026/archd?ad_name=main-menu-more
https://architecturecompetitions.com/unbuilt2026/archd?ad_source=jv-header&ad_name=hamburger_menu
https://boty.archdaily.com/?ad_name=main-menu-more
https://boty.archdaily.com/?ad_source=jv-header&ad_name=hamburger_menu
https://my.archdaily.com/audio?ad_source=mobile-bottom-nav&ad_medium=audio-button
https://my.archdaily.com/benefits?ad_source=mobile-bottom-nav&ad_medium=benefits-button
https://my.archdaily.com/maps?ad_source=mobile-bottom-nav&ad_medium=maps-button
https://www.archdaily.cl/cl
https://www.archdaily.cl/cl?ad_source=jv-header
https://www.archdaily.cn/cn
https://www.archdaily.cn/cn?ad_source=jv-header
https://www.archdaily.com
https://www.archdaily.com.br/br
https://www.archdaily.com.br/br?ad_source=jv-header
https://www.archdaily.com/advertise?ad_source=jv-header&ad_name=hamburger_menu
https://www.a

Initial attempts to scrape project links from ArchDaily revealed that the search results are dynamically rendered and not directly accessible via static HTML parsing. As a result, an alternative data source was selected to ensure efficient data collection.